In [1]:
import lightkurve as lk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import warnings
warnings.filterwarnings('ignore')

# Physical constants
G = 6.674e-11       # Gravitational constant
M_sun = 1.989e30    # Solar mass in kg
R_sun = 6.957e8     # Solar radius in meters
AU = 1.496e11       # 1 AU in meters

C:\Users\giahy\Downloads\anaconda\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
koi_table = NasaExoplanetArchive.query_criteria(
    table="koi",
    select="kepid, kepoi_name, koi_period, koi_disposition, koi_kepmag, \
            koi_prad, koi_srad, koi_smass, koi_steff",
    where="koi_disposition like 'CANDIDATE' and koi_kepmag < 14"
)

df = koi_table.to_pandas()

# Filter out missing or invalid stellar data
df_clean = df.dropna(subset=['koi_smass', 'koi_srad', 'koi_period', 'koi_steff'])
df_clean = df_clean[
    (df_clean['koi_smass'] > 0.1) &
    (df_clean['koi_srad'] > 0.1) &
    (df_clean['koi_period'] > 0) &
    (df_clean['koi_steff'] > 0)
].copy()

# Deduplicate — one row per star (brightest signal per star)
df_unique = df_clean.sort_values('koi_kepmag').drop_duplicates(
    subset='kepid', keep='first'
).reset_index(drop=True)

print(f"Total candidates:         {len(df)}")
print(f"With complete data:       {len(df_clean)}")
print(f"Unique stars:             {len(df_unique)}")

# Take 500 brightest unique stars
df_sample = df_unique.head(500).reset_index(drop=True)
print(f"Working sample:           {len(df_sample)} stars")

Total candidates:         3255
With complete data:       2301
Unique stars:             914
Working sample:           500 stars


In [3]:
def calculate_orbital_distance(period_days, stellar_mass_solar):
    T = period_days * 86400
    M = stellar_mass_solar * M_sun
    a_cubed = (G * M * T**2) / (4 * np.pi**2)
    return (a_cubed ** (1/3)) / AU

def estimate_luminosity(stellar_temp, stellar_radius_solar):
    T_sun = 5778
    return (stellar_radius_solar**2) * ((stellar_temp / T_sun)**4)

def habitable_zone(luminosity_solar):
    inner = 0.95 * np.sqrt(luminosity_solar)
    outer = 1.37 * np.sqrt(luminosity_solar)
    return inner, outer

def classify_planet(radius_earth):
    if radius_earth is None:
        return "Unknown"
    if radius_earth < 1.25:
        return "Rocky"
    elif radius_earth < 2.0:
        return "Super-Earth"
    elif radius_earth < 4.0:
        return "Mini-Neptune"
    elif radius_earth < 11.0:
        return "Gas Giant"
    else:
        return "Super-Jupiter"

In [4]:
from concurrent.futures import ThreadPoolExecutor
def check_period_match(detected, catalog, threshold=20):
    """
    Check if detected period matches catalog period
    including common harmonics (0.5x, 2x, 3x)
    Returns (verified, match_pct, harmonic_note)
    """
    harmonics = [0.5, 1.0, 1.5, 2.0, 3.0]
    best_match = 999
    best_note = "fundamental"
    
    for h in harmonics:
        match = abs(detected - catalog * h) / (catalog * h) * 100
        if match < best_match:
            best_match = match
            if h == 1.0:
                best_note = "fundamental"
            else:
                best_note = f"{h}x harmonic"
    
    verified = best_match <= threshold
    return verified, round(best_match, 2), best_note

results = []
failed = []
unverified = []

total = len(df_sample)

for idx, row in df_sample.iterrows():
    kepid = str(int(row['kepid']))
    catalog_period = row['koi_period']
    
    print(f"[{idx+1}/{total}] KIC {kepid} "
          f"(catalog period: {catalog_period:.3f}d)...", end=" ")
    
    try:
        # ── STEP 1: Download light curve ──────────────────────
        search = lk.search_lightcurve(
            f"KIC {kepid}",
            mission="Kepler",
            cadence="long"
        )
        
        if len(search) == 0:
            print("No data")
            failed.append({'kepid': kepid, 'reason': 'No light curve found'})
            continue
        
        # Download up to 3 quarters for better period detection
        n_quarters = 1
        lc_list = [search[i].download() for i in range(n_quarters)]
        lc = lk.LightCurveCollection(lc_list).stitch()
        
        # ── STEP 2: Clean ─────────────────────────────────────
        lc = lc.remove_nans().remove_outliers(sigma=3).normalize()
        
        # ── STEP 3: BLS period detection ──────────────────────
        # Cap search at 50 days — beyond this BLS needs much more data
        # Long period planets need many quarters to confirm anyway
        search_max = min(catalog_period * 2, 50)
        search_max = max(search_max, 5)          # minimum search of 5 days
        step = max(0.05, search_max / 500)        # max 500 points in range

        # Scale frequency_factor higher for long period stars
        # to prevent internal grid from becoming too large
        if catalog_period > 50:
            freq_factor = 150
        elif catalog_period > 20:
            freq_factor = 50
        else:
            freq_factor = 25
        
        pg = lc.to_periodogram(
            method="bls",
            period=np.arange(0.5, search_max, step),
            frequency_factor=freq_factor  
        )
        
        detected_period = pg.period_at_max_power.value
        bls_power = pg.max_power.value
        
        # ── STEP 4: Cross-match with NASA catalog ─────────────
        verified, period_match_pct, harmonic_note = check_period_match(
            detected_period, catalog_period
        )

        if not verified:
            print(f"Not verified (best match: {period_match_pct:.1f}%)")
            unverified.append({
                'kepid': kepid,
                'kepoi_name': row['kepoi_name'],
                'catalog_period': round(catalog_period, 4),
                'detected_period': round(detected_period, 4),
                'period_match_pct': round(period_match_pct, 2),
                'bls_power': round(bls_power, 5)
            })
            continue
        
        # ── STEP 5: Calculate physical properties ─────────────
        # Only runs if independently verified
        orbital_dist = calculate_orbital_distance(
            catalog_period,           # Use catalog period for accuracy
            row['koi_smass']
        )
        
        luminosity = estimate_luminosity(row['koi_steff'], row['koi_srad'])
        hz_inner, hz_outer = habitable_zone(luminosity)
        in_hz = hz_inner <= orbital_dist <= hz_outer
        
        planet_radius = row['koi_prad'] if pd.notna(row['koi_prad']) else None
        planet_type = classify_planet(planet_radius)

        print(f"VERIFIED ✓ "
              f"(match: {period_match_pct:.1f}% via {harmonic_note}, "
              f"dist: {orbital_dist:.3f} AU, "
              f"HZ: {in_hz})")
        
        results.append({
            'kepid': kepid,
            'kepoi_name': row['kepoi_name'],
            'catalog_period (days)': round(catalog_period, 4),
            'detected_period (days)': round(detected_period, 4),
            'period_match (%)': round(period_match_pct, 2),
            'bls_power': round(bls_power, 5),
            'orbital_dist (AU)': round(orbital_dist, 4),
            'planet_radius (R_earth)': planet_radius,
            'planet_type': planet_type,
            'stellar_mass (solar)': round(row['koi_smass'], 3),
            'stellar_temp (K)': round(row['koi_steff'], 0),
            'luminosity (solar)': round(luminosity, 4),
            'hz_inner (AU)': round(hz_inner, 4),
            'hz_outer (AU)': round(hz_outer, 4),
            'in_habitable_zone': in_hz,
            'quarters_used': n_quarters,
            'kepmag': row['koi_kepmag']
        })
        
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        failed.append({'kepid': kepid, 'reason': str(e)[:100]})
        continue

# Convert to DataFrames
results_df = pd.DataFrame(results)
unverified_df = pd.DataFrame(unverified)
failed_df = pd.DataFrame(failed)

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"Verified candidates:    {len(results_df)}")
print(f"Not verified:           {len(unverified_df)}")
print(f"Failed (no data):       {len(failed_df)}")
print(f"Verification rate:      {len(results_df)/total*100:.1f}%")

[1/500] KIC 11502218 (catalog period: 3.989d)... VERIFIED ✓ (match: 12.2% via 2.0x harmonic, dist: 0.062 AU, HZ: False)
[2/500] KIC 11180361 (catalog period: 0.533d)... VERIFIED ✓ (match: 0.1% via 3.0x harmonic, dist: 0.017 AU, HZ: False)
[3/500] KIC 5108214 (catalog period: 2.119d)... VERIFIED ✓ (match: 0.9% via fundamental, dist: 0.036 AU, HZ: False)
[4/500] KIC 4150611 (catalog period: 0.761d)... Not verified (best match: 31.4%)
[5/500] KIC 6032730 (catalog period: 96.509d)... Not verified (best match: 91.7%)
[6/500] KIC 11709006 (catalog period: 0.720d)... VERIFIED ✓ (match: 1.9% via 3.0x harmonic, dist: 0.016 AU, HZ: False)
[7/500] KIC 7022603 (catalog period: 21.127d)... VERIFIED ✓ (match: 12.1% via 0.5x harmonic, dist: 0.176 AU, HZ: False)
[8/500] KIC 3230227 (catalog period: 7.047d)... Not verified (best match: 84.4%)
[9/500] KIC 11013201 (catalog period: 7.823d)... VERIFIED ✓ (match: 10.5% via fundamental, dist: 0.098 AU, HZ: False)
[10/500] KIC 6670742 (catalog period: 72.282

In [5]:
print("\nVERIFIED CANDIDATES SUMMARY")
print("=" * 60)

if len(results_df) > 0:
    hz_df = results_df[results_df['in_habitable_zone'] == True]
    
    print(f"Total verified:         {len(results_df)}")
    print(f"In habitable zone:      {len(hz_df)}")
    print(f"\nPlanet type breakdown:")
    print(results_df['planet_type'].value_counts().to_string())
    
    print(f"\nHABITABLE ZONE CANDIDATES:")
    print("=" * 60)
    if len(hz_df) > 0:
        print(hz_df[['kepoi_name', 'catalog_period (days)',
                      'detected_period (days)', 'period_match (%)',
                      'orbital_dist (AU)', 'planet_radius (R_earth)',
                      'planet_type']].to_string())
    else:
        print("No habitable zone candidates in this sample")
else:
    print("No verified candidates — check pipeline")


VERIFIED CANDIDATES SUMMARY
Total verified:         177
In habitable zone:      0

Planet type breakdown:
planet_type
Rocky            64
Super-Jupiter    60
Super-Earth      29
Mini-Neptune     14
Gas Giant        10

HABITABLE ZONE CANDIDATES:
No habitable zone candidates in this sample


In [9]:
# ── SECTION 6: EXTENDED SELECTION BIAS ANALYSIS ─────────────

print("SELECTION BIAS — EXTENDED ANALYSIS")
print("=" * 60)

# Temperature distribution of verified stars
print("\nStellar temperature breakdown:")
bins = [0, 4000, 5000, 6000, 7000, 99999]
labels = ['<4000K (M dwarf)', '4000-5000K (K dwarf)', 
          '5000-6000K (G dwarf)', '6000-7000K (F dwarf)', '>7000K (A star)']
results_df['star_type'] = pd.cut(
    results_df['stellar_temp (K)'], 
    bins=bins, labels=labels
)
print(results_df['star_type'].value_counts().to_string())

# How far each planet is from its HZ
results_df['au_below_hz_inner'] = (
    results_df['hz_inner (AU)'] - results_df['orbital_dist (AU)']
).round(4)

print(f"\nAll 177 verified planets orbit INSIDE their star's HZ inner edge")
print(f"Mean distance below HZ inner edge: "
      f"{results_df['au_below_hz_inner'].mean():.3f} AU")
print(f"Closest planet to HZ inner edge: "
      f"{results_df['au_below_hz_inner'].min():.3f} AU short")
print(f"\nConclusion: Zero HZ candidates reflects both Kepler's")
print(f"detection bias toward short-period planets AND a sample")
print(f"dominated by hot luminous stars with distant habitable zones.")

SELECTION BIAS — EXTENDED ANALYSIS

Stellar temperature breakdown:
star_type
6000-7000K (F dwarf)    78
5000-6000K (G dwarf)    60
>7000K (A star)         22
4000-5000K (K dwarf)    16
<4000K (M dwarf)         1

All 177 verified planets orbit INSIDE their star's HZ inner edge
Mean distance below HZ inner edge: 2.388 AU
Closest planet to HZ inner edge: 0.036 AU short

Conclusion: Zero HZ candidates reflects both Kepler's
detection bias toward short-period planets AND a sample
dominated by hot luminous stars with distant habitable zones.


In [10]:
results_df.to_csv('./outputs/v4_verified_candidates.csv', index=False)
unverified_df.to_csv('./outputs/v4_unverified.csv', index=False)
failed_df.to_csv('./outputs/v4_failed.csv', index=False)

print("Saved:")
print("  outputs/v4_verified_candidates.csv")
print("  outputs/v4_unverified.csv")
print("  outputs/v4_failed.csv")

Saved:
  outputs/v4_verified_candidates.csv
  outputs/v4_unverified.csv
  outputs/v4_failed.csv
